# Test the opt-in load selector

Keep this analysis separate from evidence and model fitting. Use the built-in fictional
example or explicitly supply compatible service demand and capacity observations or
assumptions. This notebook makes no live calls or assignments.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# Fill this in if the notebook's working directory is outside the repository.
REPO_ROOT = os.environ.get("DATATHON_REPO_ROOT") or None
cwd = Path.cwd().resolve()
ROOT = Path(REPO_ROOT).expanduser().resolve() if REPO_ROOT else next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "non-ml" / "config" / "project.json").is_file()),
    None,
)
if ROOT is None or not (ROOT / "non-ml" / "config" / "project.json").is_file():
    raise RuntimeError("Set REPO_ROOT to the directory that contains non-ml/config/project.json.")
sys.path.insert(0, str(ROOT / "non-ml" / "src"))

from facility_tool.workspace import Store, read_config, show

config = read_config(ROOT)
MODE = os.environ.get("DATATHON_MODE", config["input_mode"])
STORE = Store(ROOT, MODE, config["as_of"])
print(f"Input mode: {MODE} | Evidence date: {config['as_of']}")

## Run matched comparisons
Keep the service, facility evidence, and location fixed. The opt-in selector considers
only adequately supported options of comparable scope within the extra travel limit.
Its final score can trade extra travel against a condition-specific demand/capacity
ratio. Compare only compatible unit, period, and source basis. Scenario or historical
pressure is labeled as such. Total beds alone won't stand in for dialysis slots.
For context, [Moore and Huang](https://pmc.ncbi.nlm.nih.gov/articles/PMC12107644/)
studied redistribution with known appointment availability and a travel penalty.

In [ ]:
from facility_tool.lookup import lookup
from facility_tool.recommend import simulate_spread

CONCERN = config['lookup']['service']
LOCATION = config['lookup']['location']
SCENARIO_CSV = None  # Optional fixture with facility_id, service, demand, capacity, unit, period, basis, source_url.
REQUESTS = 100  # Fictional suggestions in the optional replay. No patient records.
pressure = pd.read_csv(SCENARIO_CSV) if SCENARIO_CSV else STORE.read('pressure_scenarios', optional=True)
common = dict(concern=CONCERN, location=LOCATION, traveltime_enabled=False, pressure_inputs=pressure)
baseline = lookup(STORE, config, **common, spread_demand=False)
opt_in = lookup(STORE, config, **common, spread_demand=True)
columns = ['mode', 'name', 'rank', 'distance_km', 'extra_km_vs_baseline', 'pressure_ratio',
           'pressure_grade', 'travel_penalty', 'pressure_penalty', 'weighted_score', 'weighted_status']
show(baseline['summary'])
show(opt_in['summary'])
show(opt_in['rows'].loc[opt_in['rows']['kind'].eq('recommendation'), columns], limit=25)

## Sensitivity and a separate capacity-weighted replay
Compare first choices across multiple travel weights. The replay demonstrates how
a fixed batch of hypothetical suggestions might be spread across sites. Its output
measures recommendations shown, with no claim about visits or actual crowding.

In [ ]:
if pressure.empty:
    print('No comparable demand/capacity inputs. The ordinary ranking is available.')
else:
    sensitivity = []
    for weight in (0.25, 0.5, 0.75):
        scenario = lookup(STORE, config, **common, spread_demand=True, travel_weight=weight)
        first = scenario['rows'].loc[scenario['rows']['kind'].eq('recommendation') &
                                     scenario['rows']['rank'].eq(1)]
        sensitivity.append(first[['mode', 'name', 'extra_km_vs_baseline', 'pressure_ratio',
                                  'weighted_score', 'weighted_status']].assign(travel_weight=weight))
    show(pd.concat(sensitivity, ignore_index=True))
    if MODE == 'demo' or SCENARIO_CSV:
        base_candidates = baseline['trace'].loc[baseline['trace']['mode'].eq('confirmed')]
        show(simulate_spread(base_candidates, pressure, baseline['service'], requests=REQUESTS,
                             max_extra_km=config['lookup']['max_extra_km']))